# Predicting Smartphone Addiction — Max-Score Pipeline
### XGBoost + LightGBM + CatBoost + Deep Learning (PyTorch), stacked

You're at 0.96 and want to close the gap to the 0.97+ leaderboard scores. At this point,
three things matter more than anything else:

1. **Out-of-fold target encoding** — turns categorical/binned columns into features that
   directly encode "how often was `addicted_label`=1 for rows like this", computed WITHOUT
   leakage. This is one of the single biggest score levers on tabular competitions and
   something plain LightGBM/XGBoost/CatBoost don't do for you automatically.
2. **A deep learning model (PyTorch MLP with embeddings)** — trained on the same data. It won't
   necessarily beat the boosting models alone, but it makes mistakes in a **different pattern**
   than tree models, so combining it adds real information (not just noise) to an ensemble.
3. **Stacking with a meta-model** instead of a manual weighted average — a small logistic
   regression is trained on the out-of-fold predictions of all 4 models, so it learns exactly
   how much to trust each one (including learning to disagree with a model on parts of the data
   where it's usually wrong).

Everything is done with **5-fold CV** and **OOF (out-of-fold) evaluation**, so the ROC-AUC you
see printed is the honest number, not an optimistic one from a single split.

> This is close to what top leaderboard entries typically do differently from a "just run
> LightGBM" baseline: target encoding + multiple diverse models + a learned stack.


## 0. Environment setup

```bash
conda activate addiction
pip install optuna catboost torch --index-url https://download.pytorch.org/whl/cpu
```
(If you have a GPU and want to use it, install the matching CUDA build of torch from
https://pytorch.org/get-started/locally/ instead of the CPU-only line above.)


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
N_FOLDS = 5
N_TRIALS = 30          # Optuna trials per boosting model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


Using device: cpu


## 1. Load data

In [2]:
TRAIN_PATH = "train.csv"
TEST_PATH  = "test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print("train:", train_df.shape, " test:", test_df.shape)


train: (691369, 14)  test: (296302, 13)


## 2. Cleaning (same careful approach as before)

Impossible values -> NaN -> imputed using train-only statistics. Outliers clipped, never dropped
(every test id needs a prediction). Ordinal columns mapped to a meaningful order.


In [3]:
FEATURE_COLS = [c for c in train_df.columns if c not in ["id", "addicted_label"]]
NUMERIC_COLS = [c for c in FEATURE_COLS if train_df[c].dtype != "object"]
CATEGORICAL_COLS = [c for c in FEATURE_COLS if train_df[c].dtype == "object"]
for col in CATEGORICAL_COLS:
    print(col, "->", train_df[col].unique())


In [4]:
# ---- 4.2 Fix impossible / noisy values -> convert to NaN so they get imputed properly
# Adjust these plausible ranges if your dataset's real-world limits are different.
def clean_impossible_values(df):
    df = df.copy()

    # Convert every column used in numeric range checks, including numeric-looking text,
    # so comparisons with numeric limits are always valid.
    numeric_range_cols = [
        "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
        "work_study_hours", "sleep_hours", "weekend_screen_time",
        "notifications_per_day", "app_opens_per_day", "stress_level"
    ]
    for col in numeric_range_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "age" in df.columns:
        df.loc[(df["age"] < 5) | (df["age"] > 100), "age"] = np.nan
    for col in ["daily_screen_time_hours", "social_media_hours", "gaming_hours",
                "work_study_hours", "sleep_hours", "weekend_screen_time"]:
        if col in df.columns:
            df.loc[(df[col] < 0) | (df[col] > 24), col] = np.nan
    for col in ["notifications_per_day", "app_opens_per_day"]:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan
    if "stress_level" in df.columns:
        # assuming stress_level is on a 1-10 scale; adjust if yours differs
        df.loc[(df["stress_level"] < 0) | (df["stress_level"] > 10), "stress_level"] = np.nan
    return df

train_df = clean_impossible_values(train_df)
test_df  = clean_impossible_values(test_df)

# Refresh these lists because coercion may have changed string columns into numeric columns.
FEATURE_COLS = [c for c in train_df.columns if c not in ["id", "addicted_label"]]
NUMERIC_COLS = [c for c in FEATURE_COLS if pd.api.types.is_numeric_dtype(train_df[c])]
CATEGORICAL_COLS = [c for c in FEATURE_COLS if c not in NUMERIC_COLS]

In [5]:
# EDIT these to match your real category strings (printed above)
ORDINAL_MAPS = {
    "stress_level": {"Low": 0, "Medium": 1, "High": 2},
    "academic_work_impact": {"None": 0, "Low": 1, "Medium": 2, "High": 3},
}
for col, mapping in ORDINAL_MAPS.items():
    if col in train_df.columns and train_df[col].dtype == "object":
        train_df[col] = train_df[col].map(mapping)
        test_df[col]  = test_df[col].map(mapping)
        med = train_df[col].median()
        train_df[col] = train_df[col].fillna(med)
        test_df[col]  = test_df[col].fillna(med)

NOMINAL_COLS = [c for c in CATEGORICAL_COLS if c not in ORDINAL_MAPS]
encoders = {}
for col in NOMINAL_COLS:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    known = set(le.classes_)
    test_df[col] = test_df[col].astype(str).apply(lambda x: x if x in known else le.classes_[0])
    test_df[col] = le.transform(test_df[col])
    encoders[col] = le

for col in NUMERIC_COLS:
    low, high = train_df[col].quantile(0.01), train_df[col].quantile(0.99)
    train_df[col] = train_df[col].clip(low, high)
    test_df[col]  = test_df[col].clip(low, high)

print("Cleaning done. Ordinal fixed:", list(ORDINAL_MAPS.keys()), "| Nominal encoded:", NOMINAL_COLS)


Cleaning done. Ordinal fixed: ['stress_level', 'academic_work_impact'] | Nominal encoded: ['gender']


## 3. Feature engineering: ratios, interactions, and binning

Binning turns a smooth numeric column into buckets — this is what lets target encoding
(next section) find patterns like "people with 6-8 gaming hours AND high stress" that raw
numbers alone make harder for a model to isolate.


In [6]:
def add_features(df):
    df = df.copy()
    df["screen_time_ratio"]      = df["daily_screen_time_hours"] / 24
    df["social_media_ratio"]     = df["social_media_hours"] / (df["daily_screen_time_hours"] + 1e-3)
    df["gaming_ratio"]           = df["gaming_hours"] / (df["daily_screen_time_hours"] + 1e-3)
    df["notif_per_screen_hour"]  = df["notifications_per_day"] / (df["daily_screen_time_hours"] + 1e-3)
    df["opens_per_notif"]        = df["app_opens_per_day"] / (df["notifications_per_day"] + 1e-3)
    df["weekend_vs_weekday"]     = df["weekend_screen_time"] - df["daily_screen_time_hours"]
    df["work_sleep_ratio"]       = df["work_study_hours"] / (df["sleep_hours"] + 1e-3)
    df["sleep_deficit"]          = 8 - df["sleep_hours"]
    if "stress_level" in df.columns:
        df["stress_x_screen"]    = df["stress_level"] * df["daily_screen_time_hours"]
    # Binned versions of key numeric columns -> useful for target encoding below
    df["age_bin"] = pd.cut(df["age"], bins=[0, 13, 18, 25, 35, 50, 100], labels=False)
    df["screen_time_bin"] = pd.cut(df["daily_screen_time_hours"], bins=10, labels=False)
    return df

train_df = add_features(train_df)
test_df  = add_features(test_df)
FEATURE_COLS = [c for c in train_df.columns if c not in ["id", "addicted_label"]]
print("Total features:", len(FEATURE_COLS))


Total features: 23


## 4. Out-of-fold target encoding (the biggest lever left)

For a categorical/binned column (e.g. `gender`, `age_bin`, `screen_time_bin`), target encoding
replaces each category with "the average `addicted_label` for that category" — a very strong
signal for tree models to split on directly.

**The danger:** if you compute this average using the SAME rows you train on, you leak the
label into the feature (the model just memorizes it -> looks great on train, fails on real test).

**The fix:** compute it **out-of-fold** — for each fold, calculate the encoding using only the
OTHER folds, so a row never sees its own label reflected back at it. We also apply **smoothing**
so categories with very few rows don't get an extreme, noisy encoding.


In [7]:
TARGET_ENCODE_COLS = ["gender", "age_bin", "screen_time_bin"]
TARGET_ENCODE_COLS = [c for c in TARGET_ENCODE_COLS if c in train_df.columns]

SMOOTHING = 20   # higher = trusts the global mean more for small categories (safer, less noisy)
global_mean = train_df["addicted_label"].mean()

def smoothed_target_encode(train_fold, other_fold_full, col, global_mean, smoothing):
    """Compute a smoothed mean-target encoding for `col` from `train_fold`,
    then apply it to `other_fold_full` (which can be a validation fold or the test set)."""
    stats = train_fold.groupby(col)["addicted_label"].agg(["mean", "count"])
    smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
    return other_fold_full[col].map(smoothed).fillna(global_mean)

skf_te = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for col in TARGET_ENCODE_COLS:
    new_col = f"{col}_target_enc"
    train_df[new_col] = 0.0
    # Build the OOF encoding for train (fold-safe)
    for tr_idx, va_idx in skf_te.split(train_df, train_df["addicted_label"]):
        fold_train = train_df.iloc[tr_idx]
        train_df.loc[train_df.index[va_idx], new_col] = smoothed_target_encode(
            fold_train, train_df.iloc[va_idx], col, global_mean, SMOOTHING
        ).values
    # For test, use the encoding learned from the FULL train set (no leakage risk here,
    # since test rows are never part of the encoding computation)
    test_df[new_col] = smoothed_target_encode(train_df, test_df, col, global_mean, SMOOTHING)

FEATURE_COLS = [c for c in train_df.columns if c not in ["id", "addicted_label"]]
print("Added target-encoded columns:", [f"{c}_target_enc" for c in TARGET_ENCODE_COLS])
print("Total features now:", len(FEATURE_COLS))

X = train_df[FEATURE_COLS]
y = train_df["addicted_label"]
X_test = test_df[FEATURE_COLS]


Added target-encoded columns: ['gender_target_enc', 'age_bin_target_enc', 'screen_time_bin_target_enc']
Total features now: 26


## 5. Optuna tuning for XGBoost, LightGBM, CatBoost

Same approach as before: quick 80/20 search for good parameters, then robust 5-fold CV training
in the next section.


In [9]:
# XGBoost, LightGBM, and CatBoost require numeric or categorical dtypes.
# Normalize any object/string columns that survived the earlier cleaning cells.
X = X.copy()
X_test = X_test.copy()
for col in X.columns:
    if not pd.api.types.is_numeric_dtype(X[col]):
        train_values = X[col].astype(str)
        test_values = X_test[col].astype(str)
        categories = pd.Index(train_values.unique())
        category_codes = {value: code for code, value in enumerate(categories)}
        X[col] = train_values.map(category_codes).astype("int32")
        X_test[col] = test_values.map(category_codes).fillna(-1).astype("int32")
    else:
        X[col] = pd.to_numeric(X[col], errors="coerce")
        X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

# Any coercion-created missing values use train medians, keeping test processing train-only.
for col in X.columns:
    fill_value = X[col].median()
    X[col] = X[col].fillna(fill_value)
    X_test[col] = X_test[col].fillna(fill_value)

unsupported_dtypes = X.select_dtypes(exclude=["number", "bool", "category"]).columns.tolist()
if unsupported_dtypes:
    raise TypeError(f"Unsupported feature dtypes for boosting models: {unsupported_dtypes}")

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }
    m = XGBClassifier(**params, eval_metric="auc", random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_tr, y_tr)
    return roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])

xgb_study = optuna.create_study(direction="maximize")
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
print("Best XGBoost (quick):", xgb_study.best_value)

  0%|          | 0/30 [00:00<?, ?it/s]

Best XGBoost (quick): 0.9625846499106214


In [10]:
def lgbm_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }
    m = LGBMClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(X_tr, y_tr)
    return roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])

lgbm_study = optuna.create_study(direction="maximize")
lgbm_study.optimize(lgbm_objective, n_trials=N_TRIALS, show_progress_bar=True)
print("Best LightGBM (quick):", lgbm_study.best_value)


  0%|          | 0/30 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.034679 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4073
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [11]:
def cat_objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
    }
    m = CatBoostClassifier(**params, eval_metric="AUC", random_state=RANDOM_STATE, verbose=False)
    m.fit(X_tr, y_tr)
    return roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])

cat_study = optuna.create_study(direction="maximize")
cat_study.optimize(cat_objective, n_trials=N_TRIALS, show_progress_bar=True)
print("Best CatBoost (quick):", cat_study.best_value)


  0%|          | 0/30 [00:00<?, ?it/s]

Best CatBoost (quick): 0.9618844797417788


## 6. The Deep Learning model — PyTorch MLP with entity embeddings

Categorical columns get their own learned **embedding vectors** (a small dense representation,
better than one-hot for the network), concatenated with the (scaled) numeric columns, then fed
through a few fully-connected layers with `BatchNorm` + `Dropout` to reduce overfitting.

**Key parameters you can tune** (marked below):
- `embedding_dim` → greater = more capacity per category, but risk of overfitting on rare categories
- `hidden_dims` → wider/deeper network = more capacity, more overfitting risk, slower
- `dropout` → greater = more regularization, helps if overfitting
- `epochs` / `lr` → more epochs with a properly small learning rate = better convergence, but watch validation AUC to avoid overfitting (this notebook uses early stopping automatically)


In [14]:
# Which columns are "categorical" for embeddings vs "numeric" for direct input.
# Low-cardinality integer columns behave like categories for a neural net (gender, bins, etc.)
NN_CAT_COLS = [c for c in ["gender", "age_bin", "screen_time_bin"] + list(ORDINAL_MAPS.keys()) if c in FEATURE_COLS]
NN_NUM_COLS = [c for c in FEATURE_COLS if c not in NN_CAT_COLS]

# Embedding inputs must be non-negative integer indices. Reserve one index per column
# for missing or unseen values, including missing values from pd.cut bins.
for col in NN_CAT_COLS:
    if pd.api.types.is_numeric_dtype(train_df[col]):
        train_values = pd.to_numeric(train_df[col], errors="coerce")
        test_values = pd.to_numeric(test_df[col], errors="coerce")
        observed_max = train_values.max()
        missing_code = int(observed_max) + 1 if pd.notna(observed_max) else 0
        train_df[col] = train_values.fillna(missing_code).astype("int64")
        test_df[col] = test_values.fillna(missing_code).astype("int64")
    else:
        train_values = train_df[col].astype("string")
        test_values = test_df[col].astype("string")
        categories = pd.Index(train_values.dropna().unique())
        category_codes = {value: code for code, value in enumerate(categories)}
        missing_code = len(categories)
        train_df[col] = train_values.map(category_codes).fillna(missing_code).astype("int64")
        test_df[col] = test_values.map(category_codes).fillna(missing_code).astype("int64")

print("NN categorical inputs:", NN_CAT_COLS)
print("NN numeric inputs:", len(NN_NUM_COLS), "columns")

# Cardinalities are one greater than the largest valid embedding index.
cat_cardinalities = {
    col: int(max(train_df[col].max(), test_df[col].max())) + 1
    for col in NN_CAT_COLS
}
print("Embedding cardinalities:", cat_cardinalities)

NN categorical inputs: ['gender', 'age_bin', 'screen_time_bin', 'stress_level', 'academic_work_impact']
NN numeric inputs: 21 columns
Embedding cardinalities: {'gender': 4, 'age_bin': 5, 'screen_time_bin': 11, 'stress_level': 1, 'academic_work_impact': 1}


In [15]:
class TabularDataset(Dataset):
    """Wraps numeric + categorical columns + (optional) labels for PyTorch."""
    def __init__(self, df_num, df_cat, y=None):
        self.X_num = torch.tensor(df_num.values, dtype=torch.float32)
        self.X_cat = torch.tensor(df_cat.values, dtype=torch.long)
        self.y = torch.tensor(y.values, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return len(self.X_num)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X_num[idx], self.X_cat[idx], self.y[idx]
        return self.X_num[idx], self.X_cat[idx]


class TabularNN(nn.Module):
    def __init__(self, num_numeric, cat_cardinalities, embedding_dim=8,
                 hidden_dims=(128, 64), dropout=0.3):
        super().__init__()
        # One embedding table per categorical column
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, embedding_dim) for card in cat_cardinalities.values()
        ])
        total_input = num_numeric + embedding_dim * len(cat_cardinalities)

        layers = []
        prev_dim = total_input
        for h in hidden_dims:
            layers += [nn.Linear(prev_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1)]   # output: raw logit (sigmoid applied outside)
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat([x_num] + embs, dim=1) if embs else x_num
        return self.mlp(x).squeeze(1)   # logits


In [16]:
def train_nn_one_fold(X_num_tr, X_cat_tr, y_tr, X_num_va, X_cat_va, y_va,
                       cat_cardinalities, epochs=40, lr=1e-3, batch_size=1024,
                       embedding_dim=8, hidden_dims=(128, 64), dropout=0.3, patience=5):
    """Trains the NN on one fold with early stopping on validation ROC-AUC."""
    train_ds = TabularDataset(X_num_tr, X_cat_tr, y_tr)
    val_ds   = TabularDataset(X_num_va, X_cat_va, y_va)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=4096, shuffle=False)

    model = TabularNN(X_num_tr.shape[1], cat_cardinalities, embedding_dim, hidden_dims, dropout).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss()

    best_auc = 0.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        for xb_num, xb_cat, yb in train_loader:
            xb_num, xb_cat, yb = xb_num.to(DEVICE), xb_cat.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb_num, xb_cat)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        # Validate every epoch -> early stopping keeps us from overfitting
        model.eval()
        val_preds = []
        with torch.no_grad():
            for xb_num, xb_cat, yb in val_loader:
                xb_num, xb_cat = xb_num.to(DEVICE), xb_cat.to(DEVICE)
                logits = model(xb_num, xb_cat)
                val_preds.append(torch.sigmoid(logits).cpu().numpy())
        val_preds = np.concatenate(val_preds)
        val_auc = roc_auc_score(y_va, val_preds)

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break   # stop early: validation AUC stopped improving

    model.load_state_dict(best_state)
    return model, best_auc


def predict_nn(model, X_num, X_cat, batch_size=4096):
    ds = TabularDataset(X_num, X_cat)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    model.eval()
    preds = []
    with torch.no_grad():
        for xb_num, xb_cat in loader:
            xb_num, xb_cat = xb_num.to(DEVICE), xb_cat.to(DEVICE)
            preds.append(torch.sigmoid(model(xb_num, xb_cat)).cpu().numpy())
    return np.concatenate(preds)


In [18]:
# Scale numeric columns for the NN using fold-local preprocessing.
# Imputation and scaling are fit on each training fold to avoid validation leakage.
nn_oof = np.zeros(len(X))
nn_test_preds = np.zeros(len(X_test))
skf_nn = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
nn_fold_scores = []

X_num_all = train_df[NN_NUM_COLS].replace([np.inf, -np.inf], np.nan)
X_cat_all = train_df[NN_CAT_COLS]
X_num_test_raw = test_df[NN_NUM_COLS].replace([np.inf, -np.inf], np.nan)
X_cat_test = test_df[NN_CAT_COLS]

for fold, (tr_idx, va_idx) in enumerate(skf_nn.split(X_num_all, y), start=1):
    imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    scaler = StandardScaler()

    X_num_tr = imputer.fit_transform(X_num_all.iloc[tr_idx])
    X_num_va = imputer.transform(X_num_all.iloc[va_idx])
    X_num_test = imputer.transform(X_num_test_raw)

    X_num_tr = pd.DataFrame(scaler.fit_transform(X_num_tr), columns=NN_NUM_COLS)
    X_num_va = pd.DataFrame(scaler.transform(X_num_va), columns=NN_NUM_COLS)
    X_num_test = pd.DataFrame(scaler.transform(X_num_test), columns=NN_NUM_COLS)

    X_cat_tr = X_cat_all.iloc[tr_idx].reset_index(drop=True)
    X_cat_va = X_cat_all.iloc[va_idx].reset_index(drop=True)
    y_tr_f = y.iloc[tr_idx].reset_index(drop=True)
    y_va_f = y.iloc[va_idx].reset_index(drop=True)

    model, fold_auc = train_nn_one_fold(
        X_num_tr, X_cat_tr, y_tr_f, X_num_va, X_cat_va, y_va_f,
        cat_cardinalities,
        epochs=40,            # <- TUNE THIS. More epochs = more learning, early stopping guards overfit
        lr=1e-3,               # <- TUNE THIS. Smaller = more careful, needs more epochs
        embedding_dim=8,       # <- TUNE THIS. Greater = more capacity per category
        hidden_dims=(128, 64), # <- TUNE THIS. Wider/deeper = more capacity, more overfitting risk
        dropout=0.3,            # <- TUNE THIS. Greater = more regularization
    )
    nn_fold_scores.append(fold_auc)
    print(f"  NN fold {fold}/{N_FOLDS} AUC = {fold_auc:.5f}")

    nn_oof[va_idx] = predict_nn(model, X_num_va, X_cat_va)
    nn_test_preds += predict_nn(model, X_num_test, X_cat_test) / N_FOLDS

nn_oof_auc = roc_auc_score(y, nn_oof)
print(f"Neural Net -> mean fold AUC = {np.mean(nn_fold_scores):.5f} | OOF AUC = {nn_oof_auc:.5f}")

  NN fold 1/5 AUC = 0.93692
  NN fold 2/5 AUC = 0.93776
  NN fold 3/5 AUC = 0.93895
  NN fold 4/5 AUC = 0.93931
  NN fold 5/5 AUC = 0.93866
Neural Net -> mean fold AUC = 0.93832 | OOF AUC = 0.93831


## 7. Final CV training for XGBoost / LightGBM / CatBoost (with target-encoded features)


In [19]:
def run_cv(model_class, params, name, X, y, X_test, n_folds=N_FOLDS):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr_f, X_va_f = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr_f, y_va_f = y.iloc[tr_idx], y.iloc[va_idx]
        model = model_class(**params)
        if name == "catboost":
            model.fit(X_tr_f, y_tr_f, verbose=False)
        else:
            model.fit(X_tr_f, y_tr_f)
        fold_pred = model.predict_proba(X_va_f)[:, 1]
        oof_preds[va_idx] = fold_pred
        auc = roc_auc_score(y_va_f, fold_pred)
        scores.append(auc)
        print(f"  {name} fold {fold}/{n_folds} AUC = {auc:.5f}")
        test_preds += model.predict_proba(X_test)[:, 1] / n_folds
    oof_auc = roc_auc_score(y, oof_preds)
    print(f"{name} -> mean fold AUC = {np.mean(scores):.5f} | OOF AUC = {oof_auc:.5f}")
    return oof_auc, oof_preds, test_preds

xgb_params  = dict(xgb_study.best_params, eval_metric="auc", random_state=RANDOM_STATE, n_jobs=-1)
lgbm_params = dict(lgbm_study.best_params, random_state=RANDOM_STATE, n_jobs=-1)
cat_params  = dict(cat_study.best_params, eval_metric="AUC", random_state=RANDOM_STATE)

xgb_oof_auc, xgb_oof, xgb_test_preds   = run_cv(XGBClassifier, xgb_params, "xgboost", X, y, X_test)
lgbm_oof_auc, lgbm_oof, lgbm_test_preds = run_cv(LGBMClassifier, lgbm_params, "lightgbm", X, y, X_test)
cat_oof_auc, cat_oof, cat_test_preds    = run_cv(CatBoostClassifier, cat_params, "catboost", X, y, X_test)


  xgboost fold 1/5 AUC = 0.96234
  xgboost fold 2/5 AUC = 0.96328
  xgboost fold 3/5 AUC = 0.96340
  xgboost fold 4/5 AUC = 0.96406
  xgboost fold 5/5 AUC = 0.96294
xgboost -> mean fold AUC = 0.96321 | OOF AUC = 0.96321
[LightGBM] [Info] Number of positive: 392379, number of negative: 160716
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019700 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4061
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709424 -> initscore=0.892589
[LightGBM] [Info] Start training from score 0.892589
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

## 8. Compare all 4 models (OOF AUC)


In [20]:
results_df = pd.DataFrame({
    "model": ["xgboost", "lightgbm", "catboost", "neural_net"],
    "oof_roc_auc": [xgb_oof_auc, lgbm_oof_auc, cat_oof_auc, nn_oof_auc]
}).sort_values("oof_roc_auc", ascending=False).reset_index(drop=True)
print(results_df)

# Check how correlated the models' OOF predictions are.
# Low correlation with the neural net (e.g. < 0.97) means it's adding real diversity
# to the stack below, not just duplicating what the trees already found.
oof_matrix = pd.DataFrame({
    "xgb": xgb_oof, "lgbm": lgbm_oof, "cat": cat_oof, "nn": nn_oof
})
print("\nCorrelation between models' OOF predictions:")
print(oof_matrix.corr())


        model  oof_roc_auc
0    lightgbm     0.963710
1     xgboost     0.963206
2    catboost     0.962425
3  neural_net     0.938306

Correlation between models' OOF predictions:
           xgb      lgbm       cat        nn
xgb   1.000000  0.993324  0.991610  0.926131
lgbm  0.993324  1.000000  0.991665  0.922221
cat   0.991610  0.991665  1.000000  0.932380
nn    0.926131  0.922221  0.932380  1.000000


## 9. Stacking: train a meta-model on the 4 models' OOF predictions

Instead of guessing weights, a small **Logistic Regression** learns the optimal combination
of the 4 models directly from their out-of-fold predictions vs the true labels. This is
trained with its own internal CV to avoid overfitting the stack itself.


In [21]:
stack_X = oof_matrix.values          # (n_rows, 4) -> OOF predictions from each model
stack_y = y.values

stack_test_X = np.column_stack([xgb_test_preds, lgbm_test_preds, cat_test_preds, nn_test_preds])

# 5-fold CV for the meta-model too, so its own OOF AUC is trustworthy
skf_meta = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
meta_oof = np.zeros(len(stack_X))
meta_test_preds = np.zeros(len(stack_test_X))

for tr_idx, va_idx in skf_meta.split(stack_X, stack_y):
    meta_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    meta_model.fit(stack_X[tr_idx], stack_y[tr_idx])
    meta_oof[va_idx] = meta_model.predict_proba(stack_X[va_idx])[:, 1]
    meta_test_preds += meta_model.predict_proba(stack_test_X)[:, 1] / N_FOLDS

stack_oof_auc = roc_auc_score(stack_y, meta_oof)
print(f"Stacked meta-model OOF ROC-AUC = {stack_oof_auc:.5f}")

# See what weight the meta-model gave to each base model (fit on ALL data for interpretability)
final_meta = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(stack_X, stack_y)
for name, coef in zip(["xgb", "lgbm", "cat", "nn"], final_meta.coef_[0]):
    print(f"  learned weight for {name}: {coef:.4f}")


Stacked meta-model OOF ROC-AUC = 0.96358
  learned weight for xgb: 2.2269
  learned weight for lgbm: 3.3463
  learned weight for cat: 1.0890
  learned weight for nn: 0.6553


## 10. Save submission files

Every individual model, plus the stacked meta-model prediction (usually your best submission).


In [22]:
def save_submission(name, test_ids, preds):
    sub = pd.DataFrame({"id": test_ids, "addicted_label": preds})
    filename = f"submission_{name}.csv"
    sub.to_csv(filename, index=False)
    print(f"saved {filename}  (shape={sub.shape})")

save_submission("xgboost_cv", test_df["id"], xgb_test_preds)
save_submission("lightgbm_cv", test_df["id"], lgbm_test_preds)
save_submission("catboost_cv", test_df["id"], cat_test_preds)
save_submission("neural_net_cv", test_df["id"], nn_test_preds)
save_submission("stacked_final", test_df["id"], meta_test_preds)


saved submission_xgboost_cv.csv  (shape=(296302, 2))
saved submission_lightgbm_cv.csv  (shape=(296302, 2))
saved submission_catboost_cv.csv  (shape=(296302, 2))
saved submission_neural_net_cv.csv  (shape=(296302, 2))
saved submission_stacked_final.csv  (shape=(296302, 2))


## 11. Sanity check


In [23]:
check = pd.read_csv("submission_stacked_final.csv")
print(check.columns.tolist())
print(check.shape)
print(check["addicted_label"].between(0, 1).all())
print(check["id"].isna().sum())
check.head()


['id', 'addicted_label']
(296302, 2)
True
0


,id,addicted_label
0,691369,0.976400
1,691370,0.962996
2,691371,0.968826
3,691372,0.974508
4,691373,0.976290


## 12. If you're STILL not where you want to be

- **Try more target-encoded columns** — add interaction columns before encoding, e.g.
  `df["gender_x_stress"] = df["gender"].astype(str) + "_" + df["stress_level"].astype(str)`,
  then add it to `TARGET_ENCODE_COLS`. Combinations of 2 categories often carry signal neither
  carries alone.
- **Increase `N_TRIALS`** for Optuna and `epochs`/try different `hidden_dims` for the NN.
- **Check `oof_matrix.corr()`** from Section 8 — if all 4 models correlate above ~0.98, they're
  extracting the same signal and stacking won't help much further; the ceiling is in the data,
  not the models. At that point, revisit raw data quality: duplicate rows, contradictory rows
  (e.g. same person-like profile with different labels), or whether an important predictive
  column might be missing from the dataset entirely.
- **Try a 2nd neural net architecture** (e.g. deeper, or a residual/skip-connection MLP) and add
  it as a 5th column to the stack — more diverse models = more room for the meta-model to gain.
- Double check there's no **train/test distribution shift** (e.g. `test.csv` collected at a
  different time or population) by comparing `train_df[NUMERIC_COLS].describe()` vs
  `test_df[NUMERIC_COLS].describe()` — big mismatches hurt generalization no matter how good
  the model is.
